In [5]:
import numpy as np
import torch

In [6]:
import torchvision
from torchvision import transforms

transform = transforms.Compose([
             transforms.ToTensor(),
             transforms.Normalize((.5,), (.5,))
])
    
train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

trainLoader = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
testLoaderr = torch.utils.data.DataLoader(test, batch_size=128, shuffle=False)




In [7]:
train.classes

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [8]:
sample_img, sample_label = next(iter(trainLoader))
print(sample_label.shape)
sample_img.shape

torch.Size([128])


torch.Size([128, 1, 28, 28])

In [9]:
import torch.nn as nn
import torch.nn.functional as F
class MyCNN(nn.Module):
    def __init__(self, ):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64*5*5, 512)
        self.fc2 = nn.Linear(512, 10)



    def forward(self, x):
        print(x.shape)

        x = self.pool(F.relu(self.conv1(x)))
        print(x.shape)

        x = self.pool(F.relu(self.conv2(x)))
        print(x.shape)

        x = torch.flatten(x, 1)
        print(x.shape)
        
        x = F.relu(self.fc1(x))
        print(x.shape)

        x = self.fc2(x)
        print(x.shape)
        return x

In [10]:
model = MyCNN()

print(model)

MyCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1600, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
)


In [11]:
dummy = torch.randn(128, 1, 28, 28)
model(dummy)

torch.Size([128, 1, 28, 28])
torch.Size([128, 32, 13, 13])
torch.Size([128, 64, 5, 5])
torch.Size([128, 1600])
torch.Size([128, 512])
torch.Size([128, 10])


tensor([[-0.0105, -0.0113,  0.0888,  ..., -0.0026,  0.0090,  0.1204],
        [-0.0932,  0.0425,  0.0118,  ...,  0.0321,  0.0136,  0.1137],
        [-0.0758,  0.0430,  0.0493,  ...,  0.0317,  0.0624,  0.0787],
        ...,
        [-0.0900,  0.0031,  0.0334,  ...,  0.0693,  0.0105,  0.1130],
        [-0.0462, -0.0125,  0.0538,  ...,  0.0362,  0.0084,  0.1581],
        [-0.0475,  0.0006,  0.0594,  ...,  0.0385,  0.0227,  0.1133]],
       grad_fn=<AddmmBackward0>)

In [ ]:
#Training loop
from torch import optim
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fun = nn.CrossEntropyLoss()

for epoch in range(2):
    correct = 0
    total = 0
    for img, tar in trainLoader:
        optimizer.zero_grad()
        output = model(img)

        loss = loss_fun(pred, tar)
        loss.backward()

        _, pred = torch.max(output, 1)

        correct += (pred == tar).sum().item()
        total += tar.shape[0]
    
    acc = (correct/total) *100
    print(acc)


In [13]:
#Testing loop
total_test = 0
correct_test = 0
with torch.no_grad():
 for img, label in testLoaderr:
    output_test = model(img)
    _, pred = torch.max(output_test, 1)
    correct_test += (pred == label).sum().item()
    total_test += label.shape[0]


acc = (correct_test/total_test) * 100
print(acc)



torch.Size([128, 1, 28, 28])
torch.Size([128, 32, 13, 13])
torch.Size([128, 64, 5, 5])
torch.Size([128, 1600])
torch.Size([128, 512])
torch.Size([128, 10])
torch.Size([128, 1, 28, 28])
torch.Size([128, 32, 13, 13])
torch.Size([128, 64, 5, 5])
torch.Size([128, 1600])
torch.Size([128, 512])
torch.Size([128, 10])
torch.Size([128, 1, 28, 28])
torch.Size([128, 32, 13, 13])
torch.Size([128, 64, 5, 5])
torch.Size([128, 1600])
torch.Size([128, 512])
torch.Size([128, 10])
torch.Size([128, 1, 28, 28])
torch.Size([128, 32, 13, 13])
torch.Size([128, 64, 5, 5])
torch.Size([128, 1600])
torch.Size([128, 512])
torch.Size([128, 10])
torch.Size([128, 1, 28, 28])
torch.Size([128, 32, 13, 13])
torch.Size([128, 64, 5, 5])
torch.Size([128, 1600])
torch.Size([128, 512])
torch.Size([128, 10])
torch.Size([128, 1, 28, 28])
torch.Size([128, 32, 13, 13])
torch.Size([128, 64, 5, 5])
torch.Size([128, 1600])
torch.Size([128, 512])
torch.Size([128, 10])
torch.Size([128, 1, 28, 28])
torch.Size([128, 32, 13, 13])
torch

In [20]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=28, in_channel=1, patch_size=7, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.num_patch = (img_size//patch_size)**2

        self.projection = nn.Linear(in_channel*self.patch_size*self.patch_size, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape

        x = x.unfold(2, self.patch_size, self.patch_size) \
            .unfold(3, self.patch_size, self.patch_size)
        
        x = x.contiguous().view(B, self.num_patch , -1)
        return self.projection(x)


In [21]:
import torch
class MyVisionTransformer(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_channel=1, num_heads = 4, num_layers = 4, embed_dim=64, num_classes = 10):
        super().__init__()
        self.patch_class = PatchEmbedding(img_size, in_channel, patch_size, embed_dim)
        self.num_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.zeros(1,1,embed_dim))
        
        self.position_embed = nn.Parameter(torch.zeros(1, self.num_patches+1, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=128, batch_first=True)

        self.transformer = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)

        self.head = nn.Linear(embed_dim, num_classes)

    
    def forward(self, x):
        B = x.shape[0]

        x = self.patch_class(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        x += self.position_embed
        x = self.transformer(x)

        cls_output = x[:, 0]
        return self.head(cls_output)




In [22]:
model = MyVisionTransformer()
print(model)

MyVisionTransformer(
  (patch_class): PatchEmbedding(
    (projection): Linear(in_features=49, out_features=64, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Linear(in_features=64, out_features=10, bias=True)
)


In [28]:
fake_input = torch.randn(8, 1, 28, 28)
out = model(fake_input)
out


tensor([[ 0.1595, -0.3880,  0.8226,  0.3664, -0.1793, -0.1969,  0.1010, -0.5111,
         -0.1343, -0.7902],
        [-0.1697,  0.0841,  0.4809,  0.2178, -1.3556, -0.6821, -0.3954,  0.2059,
         -1.3129,  0.4686],
        [-0.2180, -0.1328,  0.0800, -0.0775, -0.1560, -0.4014, -0.3909, -0.9274,
         -0.1966,  0.0460],
        [-0.3585,  0.2955,  0.3255,  0.3536, -1.1370, -0.9875, -0.6571,  0.1774,
         -0.6083,  0.1125],
        [ 0.0144, -0.1087, -0.0835, -0.8992, -0.1717,  0.2313, -0.0678, -0.1831,
          0.4664, -0.4053],
        [-0.5346, -0.1641,  0.8242,  0.2279, -0.3922, -0.0224, -0.4222,  0.5505,
         -0.8084,  0.4324],
        [ 0.3400, -0.3974,  0.8263, -0.2711, -0.6784, -0.1235, -0.3637, -0.7192,
         -0.0656, -0.2261],
        [-0.5428, -0.2170,  0.1502, -0.0112,  0.0777, -0.6114, -0.5337, -0.6251,
         -0.4131,  0.4080]], grad_fn=<AddmmBackward0>)

In [ ]:
entropy = nn.CrossEntropyLoss()
optimizer_transformer = optim.Adam(model.parameters(), lr=0.001)

for _ in range(2):
    total = 0
    correct = 0
    for img, label in trainLoader:
        optimizer_transformer.zero_grad()
        output = model(img)
        loss = entropy(output, label)
        loss.backward()

        optimizer_transformer.step()
        _, pred = torch.max(output, 1)

        correct += (pred == label).sum().item() 
        total += label.size(0)
    
    acc = (correct/total) * 100



In [30]:
total_trans_test = 0
correct_trans_test = 0
with torch.no_grad():
    for img_test, label_test in testLoaderr:
        output_trans_test = model(img_test)
        
        _, pred_test = torch.max(output_trans_test, 1)

        correct_trans_test += (pred_test == label_test).sum().item()
        total_trans_test += label_test.shape(0)

acc = (correct_trans_test/total_trans_test) * 100

KeyboardInterrupt: 